In [ ]:
# -*- coding: utf-8 -*-
# ==============================================================================
# S1 : stack, profilage, construction de bronze                  (US-01 à US-09, en Python)
# Outils : pandas, SQLAlchemy + PyMySQL, YData-Profiling. Ordre des cellules = rythme de la fiche S1 (onglet Séances).
# Chaque section « # %% » est une cellule : copiez-la dans votre notebook S1_<groupe>.ipynb (notebooks/), ou lancez tout le fichier :
#     python pipelines/S1_bronze_profilage.py
# Ce fichier donne la technique ; les observations, les choix et les commentaires restent à écrire dans livrables/S1_bronze_profilage.md.
# ==============================================================================

In [6]:
# %% Paramètres et connexion
import sys
sys.path.append("../pipelines")          # notebook dans notebooks/ ; en terminal, le dossier pipelines/ est déjà dans le chemin
from pipelines.aq import *                         # eng(), reader(), lire(), mesurer(), BATCH, DATE_REF... (voir pipelines/aq.py)

# print("Lot :", BATCH, "| date de référence :", DATE_REF)
# print(lire("SELECT VERSION() AS version, @@collation_server AS collation"))     # utf8mb4_nopad_bin : comparaisons exactes

In [ ]:
# %% US-02 · Ingestion dans bronze (copie fidèle + lot, source, horodatage, empreinte de ligne)
# Règles : types texte, aucune correction, aucun rejet, quatre métadonnées ajoutées. Les tables bronze existent déjà (initdb/00_medallion.sql) : ne les recréez pas.
# Idempotence : on supprime d'abord les lignes du lot, puis on les réinsère (relancer = même résultat, sans doublon).
# Lecture avec le compte reader (lecture seule). coerce_float=False : un montant 184,10 reste 184.10 (sinon la porte B échoue).
SOURCES = [("crm", "client", "crm_client"), ("crm", "contrat", "crm_contrat"), ("crm", "type_contrat", "crm_type_contrat"), ("crm", "sinistre", "crm_sinistre"),
           ("cif", "clients", "cif_clients"), ("cif", "compte", "cif_compte"), ("cif", "pays", "cif_pays")]

def ingerer(base, table, cible):
    src = reader(base)
    with src.connect() as c:
        df = pd.read_sql(text(f"SELECT * FROM {table}"), c, coerce_float=False).astype("string")      # tout en texte
    df = df.astype(object).where(df.notna(), None)
    h = pd.util.hash_pandas_object(df.fillna("\\N").astype(str), index=False)                         # empreinte AVANT les métadonnées
    df["_row_hash"] = [f"{v:016x}" for v in h]
    df["_batch_id"] = BATCH
    df["_source_system"] = base
    with eng().begin() as c:
        c.execute(text(f"DELETE FROM bronze.{cible} WHERE _batch_id = :b"), {"b": BATCH})
        df.to_sql(cible, c, schema="bronze", if_exists="append", index=False, chunksize=10_000)
    return len(df)

executer("INSERT INTO dq.lot_ingestion (batch_id, statut) VALUES (:b, 'EN_COURS') ON DUPLICATE KEY UPDATE statut='EN_COURS', finished_at=NULL", b=BATCH)
for base, table, cible in SOURCES:
    print(f"{base}.{table:<13} -> bronze.{cible:<17} {ingerer(base, table, cible):>7} lignes")
executer("UPDATE dq.lot_ingestion SET statut='TERMINE', finished_at=NOW() WHERE batch_id = :b", b=BATCH)

In [11]:
# %% US-03 · Porte B : réconciliation source → bronze (RQ-021) et métadonnées (RQ-022)
# RQ-021 : même nombre de lignes et même somme de contrôle (SUM des CRC32 de chaque ligne), table par table. Les colonnes sont lues dans information_schema.
def somme_controle(base, table, ou=""):
    colonnes = lire("SELECT column_name FROM information_schema.columns WHERE table_schema = :s AND table_name = :t AND column_name NOT LIKE '\\_%' "
                    "AND column_name <> 'bronze_row_id' ORDER BY ordinal_position", s=base, t=table).column_name.tolist()
    liste = ", ".join(colonnes)
    r = lire(f"SELECT COUNT(*) AS n, COALESCE(SUM(CRC32(CONCAT_WS('|', {liste}))), 0) AS crc FROM {base}.{table} {ou}")
    return int(r.n[0]), int(r.crc[0])

porte_b = []
for base, table, cible in SOURCES:
    n_src, crc_src = somme_controle(base, table)
    n_brz, crc_brz = somme_controle("bronze", cible, f"WHERE _batch_id = '{BATCH}'")
    ecarts = abs(n_src - n_brz) + int(crc_src != crc_brz)
    porte_b.append(mesurer("BRONZE", "RQ-021", cible, n_src, ecarts))
    # RQ-022 : chaque ligne porte lot, source, horodatage et empreinte
    meta = lire(f"""SELECT COUNT(*) AS n, COALESCE(SUM(_batch_id IS NULL OR _batch_id = '' OR _source_system IS NULL OR _source_system = ''
                    OR _ingested_at IS NULL OR _row_hash IS NULL OR _row_hash = ''), 0) AS ko FROM bronze.{cible} WHERE _batch_id = '{BATCH}'""")
    porte_b.append(mesurer("BRONZE", "RQ-022", cible, int(meta.n[0]), int(meta.ko[0])))
print(pd.DataFrame(porte_b).sort_values(["rule_id", "table"]).to_string(index=False))      # aucun écart attendu





NameError: name 'SOURCES' is not defined

TEST :

In [12]:
def somme_controle(base, table, ou=""):
    colonnes = lire("SELECT column_name FROM information_schema.columns WHERE table_schema = :s AND table_name = :t AND column_name NOT LIKE '\\_%' "
                    "AND column_name <> 'bronze_row_id' ORDER BY ordinal_position", s=base, t=table).column_name.tolist()
    liste = ", ".join(colonnes)
    r = lire(f"SELECT COUNT(*) AS n, COALESCE(SUM(CRC32(CONCAT_WS('|', {liste}))), 0) AS crc FROM {base}.{table} {ou}")
    return int(r.n[0]), int(r.crc[0])


# Épreuve de l'écart volontaire : sur une COPIE, jamais sur bronze elle-même, et sans écrire le résultat dans dq.dq_history
executer("DROP TABLE IF EXISTS bronze.crm_client_test")
executer("CREATE TABLE bronze.crm_client_test LIKE bronze.crm_client")
executer("INSERT INTO bronze.crm_client_test SELECT * FROM bronze.crm_client")
# À vous : altérez la copie (un UPDATE ... LIMIT ..., une suppression, un doublon inséré), par exemple :
executer("""
    UPDATE bronze.crm_client_test
    SET nom = 'Mongi'
    WHERE adresse = '31 rue Anatole France'
    LIMIT 1
""")
n_t, crc_t = somme_controle("bronze", "crm_client_test")
n_s, crc_s = somme_controle("crm", "client")
print("copie contre source : écarts =", abs(n_s - n_t) + int(crc_s != crc_t), "(0 tant que la copie n'est pas altérée)")
executer("DROP TABLE bronze.crm_client_test")                      # à la fin, dans tous les cass

copie contre source : écarts = 1 (0 tant que la copie n'est pas altérée)


In [13]:
# %% US-06 · Profil complet de crm.client avec YData-Profiling (rapport HTML), puis constats chiffrés
# Le rapport montre, il ne dit pas si une valeur est valide : chaque constat se chiffre ensuite par une requête.
from ydata_profiling import ProfileReport
import os
df = lire("SELECT * FROM crm.client", reader("crm"))             # 100 000 lignes, colonnes texte telles que livrées par la source
DOSSIER_RAPPORTS = "/home/jovyan/work/reports" if os.path.isdir("/home/jovyan/work/reports") else "reports"
os.makedirs(DOSSIER_RAPPORTS, exist_ok=True)
profil = ProfileReport(df, title="AssurConnect — Profilage crm.client (100 000 lignes)", explorative=True)
profil.to_file(f"{DOSSIER_RAPPORTS}/ydata_profiling_client.html")      # à ouvrir dans un navigateur : onglets Overview, Variables, Alerts
print("rapport écrit :", f"{DOSSIER_RAPPORTS}/ydata_profiling_client.html")

# Constats chiffrés sur bronze (le lot courant)
b = lire("SELECT * FROM bronze.crm_client WHERE _batch_id = :b", b=BATCH)
b = b[[c for c in b.columns if not c.startswith("_") and c != "bronze_row_id"]]
stats = pd.DataFrame({"nuls": b.isna().sum(), "vides": (b == "").sum(), "blancs": ((b.apply(lambda s: s.str.strip()) == "") & (b != "")).sum(),
                      "distincts": b.nunique(), "long_min": b.apply(lambda s: s.str.len().min()).astype("Int64"), "long_max": b.apply(lambda s: s.str.len().max()).astype("Int64")})
print(stats)
print("\nClé client_id : lignes", len(b), "| distinctes", b.client_id.nunique(), "| nulles", int(b.client_id.isna().sum()))
print("\nDomaine de type_personne :\n", b.type_personne.value_counts(dropna=False))
print("\nPays (valeurs distinctes) :", b.pays.nunique(), "\n", b.pays.value_counts(dropna=False).head(12))
# Valeurs manquantes normales ou défauts ? nom et raison_sociale se complètent : une personne morale n'a pas de nom
print("\nni nom ni raison :", int((b.nom.isna() & b.raison_sociale.isna()).sum()), "| les deux :", int((b.nom.notna() & b.raison_sociale.notna()).sum()),
      "| nom nul :", int(b.nom.isna().sum()), "| raison nulle :", int(b.raison_sociale.isna().sum()))
print("\nBornes de date de naissance :", b.date_naissance.dropna().min(), "→", b.date_naissance.dropna().max(), "| création :", b.date_creation.dropna().min(), "→", b.date_creation.dropna().max())
# À vous : au moins cinq constats chiffrés, dont trois que le rapport révèle sans requête et un qu'il ne signale pas (par exemple le zéro initial perdu).

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 13/13 [00:03<00:00,  3.43it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

rapport écrit : /home/jovyan/work/reports/ydata_profiling_client.html
                 nuls  vides  blancs  distincts  long_min  long_max
client_id           0      0       0     100000         6         7
type_personne       0      0       0          8         6        12
nom              4277      0       0        784         3         9
prenom           4277      0       0        293         2        10
raison_sociale  95723      0       0       3229        12        27
date_naissance   4277      0       0      28413        10        10
adresse             0      0       0       7561        10        31
cp                  0      0       0        230         4         8
ville               0      0       0        172         3        24
pays                0    400       0         18         0         6
telephone        5999   2000    2000      88702         0        17
email           19365      0       0      77795        13        36
date_creation       0      0       0       539

In [16]:
# %% US-07 · Motifs de code postal, téléphone et pays, puis expressions régulières
def motif(s):
    """9 remplace un chiffre, A une lettre : on voit la forme d'une valeur, pas la valeur."""
    return s.fillna("<NULL>").str.replace(r"[0-9]", "9", regex=True).str.replace(r"[A-Za-z]", "A", regex=True)

for colonne in ["cp", "telephone", "pays"]:
    print(f"\nMotifs de {colonne} :\n", motif(b[colonne]).value_counts().head(12))

def conformite(serie, regex):
    """(testées, en échec) pour une expression régulière complète ; une valeur nulle est en échec."""
    ok = serie.fillna("").str.fullmatch(regex)
    return len(serie), int((~ok).sum())

fr = b[b.pays == "FR"]
n, ko = conformite(fr.cp, r"[0-9]{5}")
print(f"\nRQ-005 : code postal à 5 chiffres, adresses FR : {ko} écarts sur {n} ({100*ko/n:.3f} %)")           # 1 456 sur 95 437
print("A01 (zéro initial perdu) :\n", fr[fr.cp.fillna("").str.fullmatch(r"[0-9]{4}")].groupby(["cp", "ville"]).size().sort_values(ascending=False).head(10))

# Formats nationaux français, internationaux (+33/0033 et autres indicatifs),
# avec espaces, points ou tirets comme séparateurs.
regex_telephone = (
    r"(?:"
    r"0[1-9](?:[ .-]?\d{2}){4}"
    r"|(?:\+|00)[1-9]\d{0,2}(?:[ .-]?\d){6,14}"
    r")"
)
renseignes = b[b.telephone.notna() & (b.telephone.str.strip() != "")]
if regex_telephone:
    n, ko = conformite(renseignes.telephone, regex_telephone)
    print(f"téléphone : {ko} écarts sur {n} numéros renseignés")

# Pays : valide = présent dans le référentiel silver.ref_pays (chargé par load_sources.py)
ref_pays = set(lire("SELECT code_pays FROM silver.ref_pays").code_pays)
hors = b[~b.pays.isin(ref_pays)]
print(f"pays hors référentiel : {len(hors)} sur {len(b)}\n", hors.pays.value_counts(dropna=False).head(15))


Motifs de cp :
 cp
99999       97683
9999         2068
AA9A 9AA      125
9999-999      124
Name: count, dtype: int64

Motifs de telephone :
 telephone
9999999999           66638
<AAAA>                5999
+99999999999          5301
99-99-99-99-99        4267
99.99.99.99.99        4261
99 99 99 99 99        4243
+99 9 99 99 99 99     4235
                      2000
                      2000
+999999999999          338
999999999              273
9999A99999             250
Name: count, dtype: int64

Motifs de pays :
 pays
AA        97400
AAAAAA     1600
A           600
            400
Name: count, dtype: int64

RQ-005 : code postal à 5 chiffres, adresses FR : 1456 écarts sur 95437 (1.526 %)
A01 (zéro initial perdu) :
 cp    ville            
1000  Bourg-en-Bresse      92
1100  Oyonnax              77
7100  Annonay              71
1700  Miribel              69
7200  Aubenas              67
1500  Ambérieu-en-Bugey    63
1120  Montluel             53
3200  Vichy                52
3000  Moul

In [15]:
# %% US-04 · Dictionnaire du CIF (colonnes column0 à column9, anonymes)
cif = lire("SELECT * FROM bronze.cif_clients WHERE _batch_id = :b", b=BATCH)
cif = cif[[c for c in cif.columns if c.startswith("column")]]
dico = pd.DataFrame({
    "nuls": cif.isna().sum(), "distincts": cif.nunique(), "long_min": cif.apply(lambda s: s.str.len().min()).astype("Int64"), "long_max": cif.apply(lambda s: s.str.len().max()).astype("Int64"),
    "motif_dominant": cif.apply(lambda s: motif(s).value_counts().index[0]), "exemples": cif.apply(lambda s: " | ".join(s.dropna().sample(3, random_state=1).tolist()))})
print(dico.to_string())
# À vous : un nom, un type et une justification chiffrée par colonne, déduits de ces mesures et non devinés.

# Lien entre les deux tables du CIF : quelle part des valeurs d'une colonne de cif_compte se retrouve dans une colonne de cif_clients ?
compte = lire("SELECT * FROM bronze.cif_compte WHERE _batch_id = :b", b=BATCH)
compte = compte[[c for c in compte.columns if c.startswith("column")]]
liens = [(a, k, round(100 * compte[a].isin(set(cif[k].dropna())).mean(), 1)) for a in compte.columns for k in cif.columns]
print(pd.DataFrame(liens, columns=["colonne_compte", "colonne_clients", "pct_retrouvees"]).sort_values("pct_retrouvees", ascending=False).head(5))
# Une clé étrangère probable se reconnaît à un taux proche de 100 %.

          nuls  distincts  long_min  long_max      motif_dominant                                                                          exemples
column0      0      40000         7         7             A999999                                                       K186124 | K388169 | K436056
column1      0        731         3        31              AAAAAA                                                      GAILLARD | RIVIERE | LE GALL
column2    645        180         3        10              AAAAAA                                                     Nathalie | Samira | Charlotte
column3    645      21081        10        10          99/99/9999                                              15/02/1969 | 26/04/2007 | 04/12/1959
column4      0       9811        11        31  99 AAA AA AAAAAAAA               49 AVENUE DE LA LIBÉRATION | 95 rue des Écoles | 20 rue de Bretagne
column5      0        188         4         8               99999                                               